<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 3 · Bond Mathematics and Market Conventions

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch03_bond_mathematics_market_conventions.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

A quoted bond price is the product of two layers working together: present-value mathematics and market conventions. Chapter 3 prices a stylized semiannual coupon bond at a 4% yield, 0.04 per year, derives Macaulay and modified duration from present-value weights, and separates clean price, accrued interest, and dirty price. Day-count conventions are treated as data that flow through valuation, not as prose annotations.

Afterwards you can price a fixed-rate bond from its cash-flow schedule, explain why settlement uses the dirty price, and demonstrate numerically that a convention error is a valuation error.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The chapter script prices the stylized two-year semiannual coupon bond in `ch03_bond_cashflows.csv` at a 4% annual yield (0.04 per year), printing price, Macaulay duration, and modified duration. It then computes accrued interest on a 5.00 annual coupon per 100 face between 2026-01-15 and 2026-03-10 under ACT/360 and 30E/360. Note that the two accrued values differ even though the calendar interval and the coupon are identical.

In [ ]:
SCRIPT = ROOT / "code" / "ch03_bond_math_conventions.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Tie each printed number back to the convention that produced it.

- Confirm the bond prices exactly at 100.0000: with a 4% coupon on 100 face matching the 4% semiannual yield, a par price is the built-in sanity check.
- Read the durations in years: Macaulay 1.9419 is the present-value-weighted average payment time, and modified 1.9039 equals Macaulay divided by (1 + 0.04/2).
- Compare the accrued-interest pair, 0.75 under ACT/360 versus 0.7639 under 30E/360: the same 54 calendar days become 55 counted days under 30E/360, and the difference flows straight into settlement cash.
- State which figure settles cash before quoting a price: the dirty price (clean plus accrued), not the quoted clean price.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch03_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Mini-lab: Measure the day-count effect on accrued interest

Day-count conventions translate the same calendar interval into different accrual fractions, and the difference flows directly into settlement cash. The mini-lab computes accrued interest on a 5.00 annual coupon per 100 face for the 54 calendar days between the last coupon date 2026-01-15 and settlement 2026-03-10, first under ACT/360 and then under 30E/360. The dates and coupon are illustrative teaching inputs.

In [ ]:
from datetime import date

last_coupon = date(2026, 1, 15)  # previous coupon date
settlement = date(2026, 3, 10)  # settlement date
coupon = 5.00  # annual coupon per 100 face
days_act = (settlement - last_coupon).days  # actual days
ai_act360 = coupon * days_act / 360.0  # ACT/360 accrual
days_30e = (360 * (settlement.year - last_coupon.year)
            + 30 * (settlement.month - last_coupon.month)
            + min(settlement.day, 30) - min(last_coupon.day, 30))
ai_30e360 = coupon * days_30e / 360.0  # 30E/360 accrual
print(f"ACT/360: {days_act} days, accrued {ai_act360:.4f}")
print(f"30E/360: {days_30e} days, accrued {ai_30e360:.4f}")
ai_30e360 - ai_act360  # convention gap

### What to observe

With these inputs the calculation yields accrued interest of 0.7500 under ACT/360 (54 actual days over 360) and 0.7639 under 30E/360 (55 counted days over 360), so the convention choice alone changes the buyer's settlement cash by 0.0139 per 100 face. The chapter script prints the same pair: the extra day is an artifact of 30E/360 arithmetic, not of the calendar.

## Exercises

Use the mini-lab dates and the chapter script outputs to complete these checks, naming the day-count basis with every number.

1. **Move the settlement date.** Shift settlement to 2026-04-10 and recompute both accrued values. *Hint:* The ACT/360 day count grows by 31 while the 30E/360 count grows by 30.
2. **Price at a different yield.** Re-price the chapter bond's cash flows at a 5% semiannual yield and confirm the price falls below 100. *Hint:* Discount each payment with `(1 + 0.05 / 2) ** (-2 * t)`.
3. **Check the duration approximation.** Verify that a 1 basis point yield rise predicts a price change of about -0.0190 per 100 face. *Hint:* Multiply minus the modified duration 1.9039 by the price 100 and by 0.0001.
4. **Reconcile with the script.** Confirm the script's price of 100.0 follows from the 4% coupon on 100 face meeting the 4% semiannual yield. *Hint:* Coupon rate equal to yield at par is the standard price check.
5. **Explain the settlement split.** State which accrued value a buyer pays on top of the clean price under each convention. *Hint:* Dirty price = clean price plus accrued interest, computed with the bond's own day-count rule.

For more detail, see Chapter 3 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">